# P61

- Hacky notebook, but parts are here
- Images will need to be patchfied, but I'm tempted to reuse ealier ones with heatmaps. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import copy
import math
import time
import os
from tqdm import tqdm
import jax

import torch
import torch.nn.functional as F

# openpi imports
from openpi.training import config as _config
from openpi.policies import policy_config as _policy_config
from openpi.policies import aloha_policy
from openpi.shared import download
from openpi import transforms as _transforms
from openpi.shared import normalize as _normalize
from openpi.models import model as _model

print("Imports OK")

In [ ]:
joint_labels = [
    "L joint 0 (waist)", "L joint 1 (shoulder)", "L joint 2 (elbow)",
    "L joint 3 (forearm)", "L joint 4 (wrist angle)", "L joint 5 (wrist rotate)",
    "L gripper",
    "R joint 0 (waist)", "R joint 1 (shoulder)", "R joint 2 (elbow)",
    "R joint 3 (forearm)", "R joint 4 (wrist angle)", "R joint 5 (wrist rotate)",
    "R gripper",
]

In [ ]:
def frame_to_obs(frame):
    """Convert a HuggingFace dataset frame to the observation dict
    that the policy expects (post-RepackTransform format)."""
    def to_chw(img):
        arr = np.array(img)  # PIL -> HWC (H, W, 3)
        return arr.transpose(2, 0, 1)  # -> CHW (3, H, W)
    
    return {
        "images": {
            "cam_high": to_chw(frame["observation.images.cam_high"]),
            "cam_left_wrist": to_chw(frame["observation.images.cam_left_wrist"]),
            "cam_right_wrist": to_chw(frame["observation.images.cam_right_wrist"]),
        },
        "state": np.array(frame["observation.state"]),
        "prompt": "uncap the pen",
    }

## 1. Load Dataset

In [ ]:
ds = load_dataset("physical-intelligence/aloha_pen_uncap_diverse", split="train")
print(f"Total rows: {len(ds)}")
print(f"Columns: {ds.column_names}")
print(f"Episode indices: {sorted(set(ds['episode_index']))}")

In [ ]:
# Pick an episode and grab all its frames
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

In [ ]:
# Pick a frame in the middle of the episode
FRAME_IDX = 150
frame = episode[FRAME_IDX]

In [ ]:
# Display camera views
cam_names = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, cam_names):
    img = frame[f"observation.images.{name}"]
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.suptitle(f"Episode {EPISODE_IDX}, Frame {FRAME_IDX} (t={frame['timestamp']:.2f}s)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
save_dir='data/robots/p61'
os.makedirs(save_dir, exist_ok=True)

os.makedirs(save_dir+'/cam_high', exist_ok=True)
os.makedirs(save_dir+'/cam_low', exist_ok=True)
os.makedirs(save_dir+'/diffusion', exist_ok=True)

In [ ]:
for FRAME_IDX in tqdm(range(len(episode))):
    frame = episode[FRAME_IDX]
    img = frame[f"observation.images.cam_high"]
    img.save(save_dir + '/cam_high/'+str(FRAME_IDX).zfill(3)+'.png')

In [ ]:
for FRAME_IDX in tqdm(range(len(episode))):
    frame = episode[FRAME_IDX]
    img = frame[f"observation.images.cam_low"]
    img.save(save_dir + '/cam_low/'+str(FRAME_IDX).zfill(3)+'.png')

## Loopy scoopy

In [ ]:
config = _config.get_config("pi0_aloha_pen_uncap")

# Point to converted PyTorch checkpoint
checkpoint_dir = "data/robots/openpi/pi0_aloha_pen_uncap_pytorch"

print(f"Model type: {config.model.model_type}")
print(f"Action dim: {config.model.action_dim}")
print(f"Action horizon: {config.model.action_horizon}")

In [ ]:
# Create the trained policy — auto-detects PyTorch format
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(f"Policy created successfully!")
print(f"PyTorch model: {policy._is_pytorch_model}")
print(f"Device: {policy._pytorch_device}")
print(f"Number of input transforms: {len(policy._input_transform.transforms)}")
print(f"Number of output transforms: {len(policy._output_transform.transforms)}")

In [ ]:
# Quick reference inference to make sure the model works
obs_test = frame_to_obs(frame)
result_test = policy.infer(obs_test)
print(f"Actions shape: {result_test['actions'].shape}")
print(f"Inference time: {result_test['policy_timing']['infer_ms']:.1f} ms")

## 3. Reference Inference with Fixed Noise

Generate fixed noise (using numpy so we don't need JAX at all),
run `policy.infer` as a black box, and save the result as our reference.

In [ ]:
# Fix noise for reproducibility (numpy, no JAX dependency)
rng = np.random.RandomState(42)
fixed_noise = rng.randn(50, 32).astype(np.float32)
print(f"fixed_noise: {fixed_noise.shape}, dtype={fixed_noise.dtype}")
print(f"  norm={np.linalg.norm(fixed_noise):.4f}, mean={fixed_noise.mean():.4f}")

# Reference result using policy.infer as a black box
obs_ref = frame_to_obs(frame)
result_ref = policy.infer(obs_ref, noise=fixed_noise)
print(f"\nReference actions: {result_ref['actions'].shape}")
print(f"  range: [{result_ref['actions'].min():.4f}, {result_ref['actions'].max():.4f}]")

In [ ]:
device = torch.device(policy._pytorch_device)
pt_model = policy._model  # the PI0Pytorch instance
print(f"Device: {device}")
print(f"Model type: {type(pt_model).__name__}")

In [ ]:
from openpi.models_pytorch.pi0_pytorch import make_att_2d_masks
import matplotlib, jax

In [ ]:
EPISODE_IDX = 0

ep_indices = np.array(ds["episode_index"])
mask = ep_indices == EPISODE_IDX
row_indices = np.where(mask)[0]
episode = ds.select(row_indices)

global_viz_ct=0


for FRAME_IDX in range(len(episode)):
# frame = episode[FRAME_IDX]
    
    obs_manual = frame_to_obs(frame)
    inputs = policy._input_transform(obs_manual)
    
    inputs_pt = jax.tree.map(
        lambda x: torch.from_numpy(np.array(x)).to(device)[None, ...],
        inputs
    )
    
    observation = _model.Observation.from_dict(inputs_pt)
    
    images, img_masks, lang_tokens, lang_masks, state = pt_model._preprocess_observation(
        observation, train=False
    )
    
    with torch.no_grad():
        prefix_embs, prefix_pad_masks, prefix_att_masks = pt_model.embed_prefix(
            images, img_masks, lang_tokens, lang_masks
        )
    
    
    with torch.no_grad():
        # Build attention mask
        prefix_att_2d = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
        prefix_positions = torch.cumsum(prefix_pad_masks, dim=1) - 1
    
        print(f"prefix_att_2d: {prefix_att_2d.shape}")  # [1, 816, 816]
        print(f"prefix_positions: {prefix_positions.shape}")  # [1, 816]
    
        # Convert to 4D attention mask (the format HuggingFace expects)
        prefix_att_4d = pt_model._prepare_attention_masks_4d(prefix_att_2d)
        print(f"prefix_att_4d: {prefix_att_4d.shape}")  # [1, 1, 816, 816]
    
        # Set eager attention (not SDPA) so we can hook into it
        pt_model.paligemma_with_expert.paligemma.language_model.config._attn_implementation = "eager"
    
        # Forward pass: [prefix_embs, None] — only PaliGemma runs, action expert gets None
        _, kv_cache = pt_model.paligemma_with_expert.forward(
            attention_mask=prefix_att_4d,
            position_ids=prefix_positions,
            past_key_values=None,
            inputs_embeds=[prefix_embs, None],
            use_cache=True,
        )
    
    num_steps = 10
    dt = -1.0 / num_steps
    
    # Start from fixed noise
    noise_pt = torch.from_numpy(fixed_noise).to(device)[None, ...]  # [1, 50, 32]
    x_t = noise_pt.clone()
    
    # Collect trajectory for visualization
    trajectory = [x_t[0].cpu().numpy().copy()]
    velocities = []
    t = 1.0
    
    
    
    with torch.no_grad():
        for step in range(num_steps):
            timestep = torch.tensor([t], dtype=torch.float32, device=device)
    
            # === Embed suffix: state token + action/time tokens ===
            suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = pt_model.embed_suffix(
                state, x_t, timestep
            )
            if step == 0:
                print(f"suffix_embs: {suffix_embs.shape}")       # [1, 51, 1024] = 1 state + 50 action
                print(f"suffix_pad_masks: {suffix_pad_masks.shape}")
                print(f"suffix_att_masks: {suffix_att_masks.shape}")
                print(f"adarms_cond: {adarms_cond}")              # None for pi0
    
            # === Build attention mask ===
            # suffix queries attend to [prefix KV cache | suffix tokens]
            suffix_len = suffix_pad_masks.shape[1]
            batch_size = prefix_pad_masks.shape[0]
            prefix_len = prefix_pad_masks.shape[1]
    
            prefix_cross_mask = prefix_pad_masks[:, None, :].expand(batch_size, suffix_len, prefix_len)
            suffix_att_2d = make_att_2d_masks(suffix_pad_masks, suffix_att_masks)
            full_att_2d = torch.cat([prefix_cross_mask, suffix_att_2d], dim=2)
    
            if step == 0:
                print(f"full_att_2d: {full_att_2d.shape}")  # [1, 51, 816+51]
    
            # Suffix positions continue from where prefix left off
            prefix_offsets = torch.sum(prefix_pad_masks, dim=-1)[:, None]
            suffix_positions = prefix_offsets + torch.cumsum(suffix_pad_masks, dim=1) - 1
    
            # Convert to 4D mask
            full_att_4d = pt_model._prepare_attention_masks_4d(full_att_2d)
    
            # Set eager attention for action expert too
            pt_model.paligemma_with_expert.gemma_expert.model.config._attn_implementation = "eager"
    
            # === Forward pass: [None, suffix_embs] — action expert runs, PaliGemma skipped ===
            (prefix_out, suffix_out), _ = pt_model.paligemma_with_expert.forward(
                attention_mask=full_att_4d,
                position_ids=suffix_positions,
                past_key_values=kv_cache,
                inputs_embeds=[None, suffix_embs],
                use_cache=False,
                adarms_cond=[None, adarms_cond],
            )
            assert prefix_out is None
    
            # Extract velocity from last 50 tokens (action tokens, excluding state token)
            action_out = suffix_out[:, -pt_model.config.action_horizon:]
            v_t = pt_model.action_out_proj(action_out.to(torch.float32))
    
            if step == 0:
                print(f"\nv_t (velocity): {v_t.shape}")  # [1, 50, 32]
    
            # Euler step
            x_t = x_t + dt * v_t
            t = t + dt
    
            trajectory.append(x_t[0].cpu().numpy().copy())
            velocities.append(v_t[0].cpu().numpy().copy())
            print(f"  step {step}: t={t-dt:.1f}->{t:.1f}, |v_t|={v_t.norm():.2f}, |x_t|={x_t.norm():.2f}")
    
            for i in range(len(trajectory)):
                matplotlib.image.imsave(save_dir + '/diffusion/'+ str(global_viz_ct).zfill(4) +'.png', trajectory[i][:,:14].T, cmap="viridis")
    
            global_viz_ct+=1

In [ ]:
save_dir+'/diffusion'

In [ ]:
save_dir='data/robots/mar_17_3'
os.makedirs(save_dir, exist_ok=True)

for i in range(len(trajectory)):
    matplotlib.image.imsave(save_dir + '/'+ str(i).zfill(2) +'.png', trajectory[i][:,:14].T, cmap="viridis")

In [ ]:
plt.imshow(trajectory[-1][:,:14].T)

In [ ]:
# with torch.no_grad():
#     embs_list = []
#     pad_mask_list = []
#     ar_mask_list = []

#     # --- Encode each image through SigLIP ---
#     # In PyTorch: paligemma.model.get_image_features
#     # This is HuggingFace's PaliGemma vision tower (SigLIP So400m/14)
#     for i, (img, img_mask) in enumerate(zip(images, img_masks)):
#         img_emb = pt_model.paligemma_with_expert.embed_image(img)
#         bsize, num_img_tokens = img_emb.shape[:2]

#         print(f"SigLIP image {i}:")
#         print(f"  input:  {img.shape}, dtype={img.dtype}, range=[{img.min():.2f}, {img.max():.2f}]")
#         print(f"  output: {img_emb.shape}, dtype={img_emb.dtype}")

#         embs_list.append(img_emb)
#         pad_mask_list.append(img_mask[:, None].expand(bsize, num_img_tokens))
#         ar_mask_list += [0] * num_img_tokens

#     # --- Embed language tokens ---
#     # PaliGemma's embedding table + scale by sqrt(dim)
#     lang_emb = pt_model.paligemma_with_expert.embed_language_tokens(lang_tokens)
#     lang_emb_dim = lang_emb.shape[-1]
#     lang_emb = lang_emb * math.sqrt(lang_emb_dim)

#     print(f"\nLanguage embedding:")
#     print(f"  token IDs: {lang_tokens.shape}, non-padding: {lang_masks.sum()}")
#     print(f"  embeddings: {lang_emb.shape}, dtype={lang_emb.dtype}")

#     embs_list.append(lang_emb)
#     pad_mask_list.append(lang_masks)
#     ar_mask_list += [0] * lang_emb.shape[1]

#     # --- Concatenate into full prefix sequence ---
#     prefix_embs_manual = torch.cat(embs_list, dim=1)
#     prefix_pad_masks_manual = torch.cat(pad_mask_list, dim=1)

#     print(f"\nFull prefix:")
#     print(f"  embs:      {prefix_embs_manual.shape}  (3x256 image + 48 language = 816)")
#     print(f"  pad_masks: {prefix_pad_masks_manual.shape}  (valid: {prefix_pad_masks_manual.sum()})")

#     # --- Compare against reference ---
#     max_diff = (prefix_embs_manual - prefix_embs).abs().max().item()
#     print(f"\nMax diff vs embed_prefix reference: {max_diff}")

In [ ]:

print(f"\nKV cache type: {type(kv_cache).__name__}")
print(f"KV cache layers: {len(kv_cache)}")
# Each layer has (key, value) tensors
k0, v0 = kv_cache[0]
print(f"Layer 0 K: {k0.shape}, V: {v0.shape}")  # [1, num_kv_heads, 816, head_dim]

## 7. Flow Matching Denoising Loop

10 Euler steps from t=1 (noise) to t=0 (actions).
Each step embeds the suffix (state + noisy actions + timestep), runs the action expert
with the cached prefix KV, and takes an Euler step.

- man you could visualize this process as a 32x50 image, and the pull out different projections/plots - that would be kinda interesting!

In [ ]:
num_steps = 10
dt = -1.0 / num_steps

# Start from fixed noise
noise_pt = torch.from_numpy(fixed_noise).to(device)[None, ...]  # [1, 50, 32]
x_t = noise_pt.clone()

# Collect trajectory for visualization
trajectory = [x_t[0].cpu().numpy().copy()]
velocities = []
t = 1.0

with torch.no_grad():
    for step in range(num_steps):
        timestep = torch.tensor([t], dtype=torch.float32, device=device)

        # === Embed suffix: state token + action/time tokens ===
        suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = pt_model.embed_suffix(
            state, x_t, timestep
        )
        if step == 0:
            print(f"suffix_embs: {suffix_embs.shape}")       # [1, 51, 1024] = 1 state + 50 action
            print(f"suffix_pad_masks: {suffix_pad_masks.shape}")
            print(f"suffix_att_masks: {suffix_att_masks.shape}")
            print(f"adarms_cond: {adarms_cond}")              # None for pi0

        # === Build attention mask ===
        # suffix queries attend to [prefix KV cache | suffix tokens]
        suffix_len = suffix_pad_masks.shape[1]
        batch_size = prefix_pad_masks.shape[0]
        prefix_len = prefix_pad_masks.shape[1]

        prefix_cross_mask = prefix_pad_masks[:, None, :].expand(batch_size, suffix_len, prefix_len)
        suffix_att_2d = make_att_2d_masks(suffix_pad_masks, suffix_att_masks)
        full_att_2d = torch.cat([prefix_cross_mask, suffix_att_2d], dim=2)

        if step == 0:
            print(f"full_att_2d: {full_att_2d.shape}")  # [1, 51, 816+51]

        # Suffix positions continue from where prefix left off
        prefix_offsets = torch.sum(prefix_pad_masks, dim=-1)[:, None]
        suffix_positions = prefix_offsets + torch.cumsum(suffix_pad_masks, dim=1) - 1

        # Convert to 4D mask
        full_att_4d = pt_model._prepare_attention_masks_4d(full_att_2d)

        # Set eager attention for action expert too
        pt_model.paligemma_with_expert.gemma_expert.model.config._attn_implementation = "eager"

        # === Forward pass: [None, suffix_embs] — action expert runs, PaliGemma skipped ===
        (prefix_out, suffix_out), _ = pt_model.paligemma_with_expert.forward(
            attention_mask=full_att_4d,
            position_ids=suffix_positions,
            past_key_values=kv_cache,
            inputs_embeds=[None, suffix_embs],
            use_cache=False,
            adarms_cond=[None, adarms_cond],
        )
        assert prefix_out is None

        # Extract velocity from last 50 tokens (action tokens, excluding state token)
        action_out = suffix_out[:, -pt_model.config.action_horizon:]
        v_t = pt_model.action_out_proj(action_out.to(torch.float32))

        if step == 0:
            print(f"\nv_t (velocity): {v_t.shape}")  # [1, 50, 32]

        # Euler step
        x_t = x_t + dt * v_t
        t = t + dt

        trajectory.append(x_t[0].cpu().numpy().copy())
        velocities.append(v_t[0].cpu().numpy().copy())
        print(f"  step {step}: t={t-dt:.1f}->{t:.1f}, |v_t|={v_t.norm():.2f}, |x_t|={x_t.norm():.2f}")

In [ ]:
import matplotlib

In [ ]:
trajectory[0].shape

In [ ]:
save_dir='data/robots/mar_17_3'
os.makedirs(save_dir, exist_ok=True)

for i in range(len(trajectory)):
    matplotlib.image.imsave(save_dir + '/'+ str(i).zfill(2) +'.png', trajectory[i][:,:14].T, cmap="viridis")

In [ ]:
plt.imshow(trajectory[0][:,:14].T)

In [ ]:
plt.imshow(trajectory[-2][:,:14].T)

In [ ]:
plt.imshow(trajectory[-1][:,:14].T)

In [ ]:
plt.imshow(trajectory[-1][:,:14].T)
cbar=plt.colorbar()
cbar.set_ticks([])
plt.savefig(save_dir+'/colorbar_to_grab.png', dpi=250, transparent=True)

In [ ]:
CHILL_BROWN = '#948979'

fig=plt.figure(0, (12, 3), facecolor='k')
ax=fig.add_subplot(111, facecolor='k')
plt.plot(trajectory[-1][:,8], c='#FFD35A', linewidth=2)

ax.tick_params(colors=CHILL_BROWN, labelsize=16)
for spine in ax.spines.values():
    spine.set_edgecolor(CHILL_BROWN)
ax.xaxis.label.set_color(CHILL_BROWN)
ax.yaxis.label.set_color(CHILL_BROWN)
plt.ylim(0.2, 0.6)

plt.savefig(save_dir+'/shoulder_timeseries_2.png', dpi=250)

In [ ]:
# Apply the same output transforms to our manually denoised actions
outputs_manual = {
    "state": np.asarray(inputs_pt["state"][0].cpu()),
    "actions": x_t[0].cpu().numpy(),
}
outputs_manual = policy._output_transform(outputs_manual)

predicted_actions = outputs_manual["actions"]

# Gather ground truth actions for the next action_horizon frames
action_horizon = predicted_actions.shape[0]
n_gt = min(action_horizon, len(episode) - FRAME_IDX)

gt_actions = np.array([episode[FRAME_IDX + t]["action"] for t in range(n_gt)])
print(f"Ground truth actions: {gt_actions.shape}")
print(f"Predicted actions: {predicted_actions.shape}")

# Plot predicted vs ground truth for each joint
fig, axes = plt.subplots(7, 2, figsize=(16, 28), sharex=True)
time_steps = np.arange(n_gt)

for arm_idx, arm_name in enumerate(["Left", "Right"]):
    offset = arm_idx * 7
    for joint_idx in range(7):
        ax = axes[joint_idx, arm_idx]
        dim = offset + joint_idx
        
        ax.plot(time_steps, gt_actions[:n_gt, dim], 'b-', linewidth=2, label='Ground Truth', alpha=0.8)
        ax.plot(time_steps, predicted_actions[:n_gt, dim], 'r--', linewidth=2, label='Predicted', alpha=0.8)
        
        ax.set_ylabel(joint_labels[dim].replace(f"{arm_name[0]} ", ""), fontsize=9)
        ax.grid(True, alpha=0.3)
        if joint_idx == 0:
            ax.set_title(f"{arm_name} Arm", fontsize=12, fontweight='bold')
            ax.legend(fontsize=8)
        if joint_idx == 6:
            ax.set_xlabel("Timestep")

plt.suptitle(f"PyTorch Model: Predicted vs Ground Truth Actions (from frame {FRAME_IDX})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
